# 17.7 策略梯度 / Policy Gradient (REINFORCE)

**中文**：到目前为止(Q-learning、DQN)都是**基于价值(value-based)** 的:先学 $Q(s,a)$，再"贪心取 max"间接得到策略。本节换一条完全不同的路——**基于策略(policy-based)**:**直接**把策略 $\pi(a|s;\theta)$ 参数化成一个神经网络，用**梯度上升**直接优化"期望回报"。不再绕道价值函数。这条路是现代 RL 主流(Actor-Critic、PPO、RLHF 训练大模型)的**根基**。
**English**: So far (Q-learning, DQN) we were **value-based**: learn $Q(s,a)$, then "greedily take the max" to get the policy indirectly. Now a completely different route — **policy-based**: **directly** parameterize the policy $\pi(a|s;\theta)$ as a neural network and optimize expected return by **gradient ascent**. No detour through a value function. This route is the **foundation** of mainstream modern RL (Actor-Critic, PPO, RLHF for training LLMs).

---

**中文**：目标很直白:最大化**期望回报** $J(\theta)=\mathbb E_{\tau\sim\pi_\theta}[G(\tau)]$($\tau$ 是一条轨迹)。难点:$G$ 依赖环境(不可导)，怎么对 $\theta$ 求梯度？**策略梯度定理**给出漂亮的答案:
**English**: The objective is direct: maximize **expected return** $J(\theta)=\mathbb E_{\tau\sim\pi_\theta}[G(\tau)]$ ($\tau$ a trajectory). The catch: $G$ depends on the environment (non-differentiable) — how to take a gradient w.r.t. $\theta$? The **policy gradient theorem** gives an elegant answer:

$$\nabla_\theta J(\theta)=\mathbb E_{\pi_\theta}\Big[\sum_t \nabla_\theta\log\pi_\theta(a_t|s_t)\cdot G_t\Big]$$

**中文**：直觉极其简单:**"哪个动作带来了高回报，就提高它的概率;带来低回报，就降低它的概率。"** 用回报 $G_t$ 给"提高该动作对数概率"的梯度**加权**。这就是 **REINFORCE** 算法:玩一整局→算每步的回报→按 $\nabla\log\pi\cdot G$ 更新。整个过程**不需要知道环境模型、也不需要价值函数**。
**English**: The intuition is stunningly simple: **"if an action led to high return, increase its probability; if low, decrease it."** Weight the gradient of "increase this action's log-probability" by the return $G_t$. This is **REINFORCE**: play a full episode → compute each step's return → update by $\nabla\log\pi\cdot G$. All **without a model of the environment or a value function**.

**中文**：但 REINFORCE 有个大问题——**方差极大**。以 CartPole 为例,每步奖励都是 +1,所以**所有** $G_t$ 都是正数——于是每个被采样到的动作都被"奖励"(概率被提高),只是幅度不同,学起来又慢又抖。解法:减一个**基线(baseline) $b(s)$**(通常是状态价值 $V(s)$),得到**优势(advantage) $A_t=G_t-b(s_t)$**——它衡量"这个动作比平均好多少"。比平均好就提高、比平均差就降低。减基线**不改变梯度的期望(无偏)**,却能**大幅降低方差**。
**English**: But REINFORCE has a big problem — **very high variance**. In CartPole, every step's reward is +1, so **all** $G_t$ are positive — meaning every sampled action gets "rewarded" (its probability raised), just by different amounts, making learning slow and jittery. The fix: subtract a **baseline $b(s)$** (usually the state value $V(s)$), giving the **advantage $A_t=G_t-b(s_t)$** — "how much better than average is this action." Better than average → raise; worse → lower. Subtracting a baseline **leaves the gradient's expectation unchanged (unbiased)** but **greatly reduces variance**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 现代 RL 根基必考）**
> **中文**：**策略梯度=直接参数化 π(a|s;θ), 梯度上升最大化期望回报**。策略梯度定理:$\nabla J=E[\nabla\log\pi(a|s)\cdot G]$(高回报动作↑概率)。**REINFORCE**=蒙特卡洛策略梯度(用整回合回报)。**vs 价值方法**:①能直接输出**随机策略**(适合部分可观测、需要探索); ②天然处理**连续动作**(输出高斯均值方差); ③直接优化目标。**软肋**:①**高方差**(→减基线/优势, 无偏降方差); ②**on-policy 样本效率低**(数据用一次就扔); ③只保证收敛到**局部**最优。基线用 V(s)→就成了 **Actor-Critic**(下节)。RLHF 训练大模型用的 PPO 就是策略梯度的后裔。
> **English**: **Policy gradient = directly parameterize π(a|s;θ), gradient-ascend to maximize expected return**. PG theorem: $\nabla J=E[\nabla\log\pi(a|s)\cdot G]$ (raise the probability of high-return actions). **REINFORCE** = Monte-Carlo policy gradient (uses full-episode returns). **vs value methods**: ① can output a **stochastic policy** directly (good for partial observability / needed exploration); ② naturally handles **continuous actions** (output a Gaussian mean/variance); ③ optimizes the objective directly. **Weaknesses**: ① **high variance** (→ subtract a baseline/advantage, unbiased variance reduction); ② **on-policy sample inefficiency** (data used once then discarded); ③ only guaranteed **local** optimum. Using V(s) as the baseline makes it **Actor-Critic** (next). PPO — used for RLHF of LLMs — descends from policy gradients.


In [ ]:

# ============================================================
# 环境 CartPole + 策略网络 / CartPole + policy network
# 中文:复用 17.6 的 CartPole。策略网络:4维状态 -> softmax 输出两个动作的概率(随机策略)。
# English: reuse 17.6's CartPole. Policy net: 4-dim state -> softmax over 2 actions (a stochastic policy).
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, random, time, matplotlib.pyplot as plt
def set_seed(x): torch.manual_seed(x); np.random.seed(x); random.seed(x)
class CartPole:
    g=9.8; mc=1.0; mp=0.1; l=0.5; fm=10.0; tau=0.02
    def reset(s): s.state=np.random.uniform(-0.05,0.05,4); s.steps=0; return s.state.copy()
    def step(s,a):
        x,xd,th,thd=s.state; force=s.fm if a==1 else -s.fm
        ct,st=np.cos(th),np.sin(th); tot=s.mc+s.mp
        temp=(force+s.mp*s.l*thd**2*st)/tot
        thacc=(s.g*st-ct*temp)/(s.l*(4/3-s.mp*ct**2/tot)); xacc=temp-s.mp*s.l*thacc*ct/tot
        x+=s.tau*xd; xd+=s.tau*xacc; th+=s.tau*thd; thd+=s.tau*thacc
        s.state=np.array([x,xd,th,thd]); s.steps+=1
        done=abs(x)>2.4 or abs(th)>12*np.pi/180 or s.steps>=500
        return s.state.copy(),1.0,done

class PolicyNet(nn.Module):                                   # 输出动作概率分布 / outputs an action distribution
    def __init__(s): super().__init__(); s.f=nn.Sequential(nn.Linear(4,128),nn.ReLU(),nn.Linear(128,2))
    def forward(s,x): return F.softmax(s.f(x),dim=-1)
class ValueNet(nn.Module):                                    # 基线:估计状态价值 V(s) / baseline value net
    def __init__(s): super().__init__(); s.f=nn.Sequential(nn.Linear(4,128),nn.ReLU(),nn.Linear(128,1))
    def forward(s,x): return s.f(x).squeeze(-1)

def discounted_returns(rews, gamma=0.99):                     # 从后往前算每步折扣回报 G_t / returns
    G=0; out=[]
    for r in reversed(rews): G=r+gamma*G; out.append(G)
    return out[::-1]
print("CartPole + 策略网络就绪 / ready")


**中文**：先实现**原始 REINFORCE**(不减基线):玩一整局，采样动作、记录 $\log\pi$ 和奖励;回合结束算每步回报 $G_t$;损失 $=-\sum_t \log\pi(a_t|s_t)\cdot G_t$(负号因为要梯度**上升**),反向传播更新策略。
**English**: First **vanilla REINFORCE** (no baseline): play a full episode, sampling actions and recording $\log\pi$ and rewards; at episode end compute each step's return $G_t$; loss $=-\sum_t \log\pi(a_t|s_t)\cdot G_t$ (negative because we gradient-**ascend**), backprop to update the policy.


In [ ]:

# ============================================================
# 原始 REINFORCE 与 REINFORCE+基线 / vanilla REINFORCE and with baseline
# ============================================================
def train_reinforce(episodes=500, use_baseline=False, gamma=0.99, seed=0):
    set_seed(seed); env=CartPole(); pi=PolicyNet(); opt=torch.optim.Adam(pi.parameters(), lr=2e-3)
    if use_baseline: V=ValueNet(); optV=torch.optim.Adam(V.parameters(), lr=2e-3)
    lengths=[]; adv_snapshot=None
    for ep in range(episodes):
        s=env.reset(); done=False; logps=[]; rews=[]; states=[]
        while not done:                                       # 玩一整局, 采样动作 / roll out one episode
            st=torch.tensor(s,dtype=torch.float32); probs=pi(st)
            dist=torch.distributions.Categorical(probs); a=dist.sample()
            logps.append(dist.log_prob(a)); states.append(st)
            s,r,done=env.step(int(a)); rews.append(r)
        G=torch.tensor(discounted_returns(rews,gamma),dtype=torch.float32); lengths.append(len(rews))
        if use_baseline:
            S=torch.stack(states); v=V(S)                     # 估计每个状态的基线 V(s) / baseline
            adv=(G - v).detach()                              # 优势 = 回报 - 基线 / advantage
            optV.zero_grad(); F.mse_loss(v, G).backward(); optV.step()   # 训练价值网络拟合 G / fit V to returns
        else:
            adv=G                                             # 原始:直接用回报(所有都为正)/ raw returns
        if ep==episodes-1: adv_snapshot=adv.detach().numpy()  # 记录最后一局的优势分布 / snapshot
        loss=-(torch.stack(logps)*adv).sum()                  # 策略梯度损失 / policy-gradient loss
        opt.zero_grad(); loss.backward(); opt.step()
    return lengths, adv_snapshot

t=time.time(); van_len, van_adv = train_reinforce(500, use_baseline=False)
bl_len, bl_adv = train_reinforce(500, use_baseline=True)
print(f"原始 REINFORCE      最后50回合平均存活: {np.mean(van_len[-50:]):.0f}")
print(f"REINFORCE + 基线    最后50回合平均存活: {np.mean(bl_len[-50:]):.0f}")
print(f"用时 {time.time()-t:.0f}s | 满分 500, 随机 ~20")


**中文**：REINFORCE+基线明显更快更稳。为什么？关键在**优势的分布**。原始 REINFORCE 用的回报 $G_t$ **全是正数**(CartPole 每步+1)——于是**每个动作都被"奖励"**(概率被推高),模型很难分辨"到底哪个动作更好"。减去基线后,优势 $A_t=G_t-V(s_t)$ **围绕 0 分布**——比平均好的动作 $A>0$(概率↑)、比平均差的 $A<0$(概率↓),信号清晰得多。下面把两者的分布画出来。
**English**: REINFORCE+baseline is clearly faster and more stable. Why? The key is the **distribution of the advantage**. Vanilla REINFORCE uses returns $G_t$ that are **all positive** (CartPole gives +1 each step) — so **every action gets "rewarded"** (its probability pushed up), making it hard to tell "which action is actually better." After subtracting the baseline, the advantage $A_t=G_t-V(s_t)$ is **centered around 0** — better-than-average actions get $A>0$ (probability ↑), worse ones $A<0$ (probability ↓), a much clearer signal. Let's plot both distributions.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
def smooth(x,k=15): return np.convolve(x,np.ones(k)/k,mode="valid")
fig,ax=plt.subplots(1,3,figsize=(17,4.6))
# ① 学习曲线对比 / learning curves
ax[0].plot(smooth(van_len),color="#C44E52",lw=2,label=f"原始 vanilla ({np.mean(van_len[-50:]):.0f})")
ax[0].plot(smooth(bl_len),color="#4C72B0",lw=2,label=f"+基线 baseline ({np.mean(bl_len[-50:]):.0f})")
ax[0].axhline(500,ls=":",color="gray"); ax[0].set_title("REINFORCE:基线加速学习 / baseline speeds up")
ax[0].set_xlabel("episode"); ax[0].set_ylabel("存活步数 survival"); ax[0].legend(fontsize=9)
# ② 原始回报分布(全正)/ raw returns (all positive)
ax[1].hist(van_adv,bins=30,color="#C44E52",edgecolor="white")
ax[1].axvline(0,color="k",ls="--"); ax[1].set_title("原始:回报G全为正→所有动作都被奖励\nraw G all positive")
ax[1].set_xlabel("加权信号 weight (=G)"); ax[1].set_ylabel("count")
# ③ 优势分布(围绕0)/ advantage centered around 0
ax[2].hist(bl_adv,bins=30,color="#4C72B0",edgecolor="white")
ax[2].axvline(0,color="k",ls="--"); ax[2].set_title("加基线:优势A围绕0→有升有降\nadvantage centered at 0")
ax[2].set_xlabel("加权信号 weight (=A=G-V)"); ax[2].set_ylabel("count")
plt.tight_layout(); plt.savefig("/tmp/rl07_viz.png",dpi=80); plt.show()
print(f"原始加权信号均值 {van_adv.mean():.1f}(远大于0, 全在推高概率); 加基线后均值 {bl_adv.mean():.2f}(围绕0)")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **策略梯度直接优化策略、且天生随机**:REINFORCE 不学价值、不查表,直接调策略网络的参数让"高回报动作更常被选"。它输出的是一个**随机策略**(概率分布),这在需要探索、或最优策略本身就该随机(如石头剪刀布、部分可观测环境)时是巨大优势——这是价值方法(确定性贪心)做不到的。
2. **基线 = 无偏的方差缩减**:原始 REINFORCE 的加权信号(回报)**均值远大于 0**(本例 ~几十),意味着"每个动作都在被推高概率",区分度差、方差大;减去基线后信号**围绕 0**,只把"比预期好"的动作往上推。数学上减基线**不改变梯度期望**(因为 $E[\nabla\log\pi\cdot b]=0$),纯赚方差。学习曲线因此更快更稳(本例 193→487)。
3. **诚实的局限**:①**on-policy**——每局数据**只能用一次**(更新完策略,旧数据就"过期"),样本效率远低于 DQN 的经验回放;②**方差仍大**(尤其长回合),训练抖;③只保证**局部**最优。这三点正是接下来 **Actor-Critic(用 V(s) 当基线+自举)** 和 **PPO(限制更新步长+多次复用数据)** 要改进的。

**English**:
1. **Policy gradient optimizes the policy directly and is inherently stochastic**: REINFORCE learns no value and reads no table — it tunes the policy network so "high-return actions get chosen more often." It outputs a **stochastic policy** (a distribution), a big advantage when exploration is needed or the optimal policy is genuinely random (rock-paper-scissors, partial observability) — something value methods (deterministic greedy) can't do.
2. **The baseline = unbiased variance reduction**: vanilla REINFORCE's weighting signal (the return) has a **mean far above 0** (~tens here), meaning "every action's probability is being pushed up," giving poor discrimination and high variance; subtracting the baseline **centers the signal at 0**, pushing up only "better-than-expected" actions. Mathematically, subtracting a baseline **doesn't change the gradient's expectation** (since $E[\nabla\log\pi\cdot b]=0$) — pure variance gain. Hence faster, more stable learning (193→487 here).
3. **Honest limits**: ① **on-policy** — each episode's data can be used **only once** (after updating the policy, old data is "stale"), far less sample-efficient than DQN's replay; ② **variance is still high** (especially long episodes), training jitters; ③ only guaranteed **local** optimum. These three are exactly what **Actor-Critic (V(s) as baseline + bootstrapping)** and **PPO (bounded update steps + reusing data)** improve next.

> 💼 **实战视角 / Practical angle**
> **中文**:策略梯度是现代 RL 的**主干**:①**连续控制**(机器人、自动驾驶)几乎都用策略方法(输出连续动作分布); ②**RLHF 对齐大模型** 用的 PPO 就是策略梯度的后裔(把"人类偏好奖励"当回报优化语言策略); ③游戏 AI(星际、Dota)。**工程要点**:回报要**标准化/减基线**降方差; 熵正则鼓励探索; 学习率要小(策略突变会崩)。面试金句:*"REINFORCE 直接对期望回报做梯度上升——高回报动作提概率; 但回报全正/高方差, 减基线得到优势(无偏降方差); on-policy 样本效率低——于是有了 Actor-Critic 和 PPO。"*
> **English**: Policy gradients are the **backbone** of modern RL: ① **continuous control** (robots, self-driving) almost always uses policy methods (output a continuous action distribution); ② **RLHF alignment of LLMs** uses PPO, a policy-gradient descendant (optimize the language policy against a "human-preference reward"); ③ game AI (StarCraft, Dota). **Engineering**: standardize returns / subtract a baseline to cut variance; entropy regularization encourages exploration; keep the learning rate small (abrupt policy shifts collapse). Interview line: *"REINFORCE gradient-ascends expected return directly — raise the probability of high-return actions; but returns are all-positive/high-variance, so subtract a baseline for the advantage (unbiased variance reduction); on-policy is sample-inefficient — hence Actor-Critic and PPO."*

---
### 小结 / Summary
- **中文**:策略梯度直接参数化 π 并梯度上升期望回报; REINFORCE=用整回合回报的蒙特卡洛策略梯度。
- **English**: Policy gradient directly parameterizes π and gradient-ascends expected return; REINFORCE = Monte-Carlo policy gradient using full-episode returns.
- **中文**:核心问题是高方差; 减基线→优势(无偏降方差), 学习更快更稳。
- **English**: The core problem is high variance; subtract a baseline → advantage (unbiased variance reduction), learning faster and more stable.
- **中文**:能输出随机策略、天然处理连续动作; 但 on-policy 样本效率低——引出 Actor-Critic 与 PPO。
- **English**: Outputs stochastic policies and handles continuous actions naturally; but on-policy is sample-inefficient — motivating Actor-Critic and PPO.
